In [43]:
import pandas as pd
import numpy as np

In [44]:
# Ijara narxi modeli: ŷ = 0.05·x + 1, bunda x = kvartira maydoni (m²), ŷ = oylik ijara (mln so'm).
# Test to'plami: x = [40, 60, 80], haqiqiy ijara y = [3.2, 3.8, 5.4] mln so'm.

In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_test = np.array([40, 60, 80])
y_test = np.array([3.2, 3.8, 5.4])

y_pred = 0.05 * X_test + 1.0
print(f"Bashorat (y_hat): {y_pred}")

mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:  {mae:.4f}")

mse = mean_squared_error(y_test, y_pred)
print(f"MSE:  {mse:.4f}")

rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.4f}")

r2 = r2_score(y_test, y_pred)
print(f"R²:   {r2:.4f}")

Bashorat (y_hat): [3. 4. 5.]
MAE:  0.2667
MSE:  0.0800
RMSE: 0.2828
R²:   0.9072


In [46]:
from sklearn.linear_model import LinearRegression

x = np.array([40, 60, 80]).reshape(-1, 1)
y = np.array([3.2, 3.8, 5.4])

lr = LinearRegression()
lr.fit(x, y)

print("coef : ", lr.coef_[0])
print("intercept: ", lr.intercept_)

coef :  0.055
intercept:  0.8333333333333339


In [47]:
# M2. Bank krediti: qaytaradimi?    Logistik regressiya  •  10 ball

# Model: z = 0.8·x1 − 0.5·x2 − 1, p = σ(z) = 1 / (1 + e^(−z)). p = mijozning kreditni qaytarmaslik ehtimoli.
# x1 = oldingi kreditlardagi kechikishlar soni, x2 = oylik daromad (mln so'm).
# Mijoz A: x1 = 3, x2 = 2.   Mijoz B: x1 = 0, x2 = 6.


# 1.  Ikkala mijoz uchun z va p ni toping.  [3 ball]


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# Mijoz A: x1 = 3, x2 = 2.
z_A = 0.8 * 3 - 0.5 * 2 - 1
print(f"Mijoz A uchun z = {z_A:.2f}")

# Mijoz B: x1 = 0, x2 = 6.
z_B = 0.8 * 0 - 0.5 * 6 - 1
print(f"Mijoz B uchun z = {z_B:2f}")


def p(z):
    return sigmoid(z)


print(f"Mijoz A ning kreditni qaytarmaslik ehtimoli -> {p(z_A):.2f}")
print(f"Mijoz B ning kreditni qaytarmaslik ehtimoli -> {p(z_B):.2f}")


# 2. Chegara 0.5 bo'lsa, har bir mijoz uchun bashorat qaysi sinf (1 = qaytarmaydi)?  [1 ball]

threshold = 0.5

p_z_A = p(z_A)
if p_z_A > threshold:
    print(f"Mijoz A kreditni qaytarmaydi yani 1")
else:
    print(f"Mijoz A kreditni qaytaradi yani 0")

p_z_B = p(z_B)
if p_z_B > threshold:
    print(f"Mijoz B kreditni qaytarmaydi yani 1")
else:
    print(f"Mijoz B kreditni qaytaradi yani 0")

# 3.  5 ta mijozning ehtimollari p = [0.15, 0.35, 0.45, 0.62, 0.81], haqiqiy y = [0, 1, 0, 1, 1]. Chegara 0.5 va 0.3 uchun TP, FP, FN va recall'ni toping.  [3 ball]
from sklearn.metrics import confusion_matrix, recall_score

p = np.array([0.15, 0.35, 0.45, 0.62, 0.81])
y_true = np.array([0, 1, 0, 1, 1])


def evaluate_threshold(probabilities, true_labels, threshold):
    y_pred = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(true_labels, y_pred).ravel()

    recall = recall_score(true_labels, y_pred)

    print(f"--- Chegara: {threshold} ---")
    print(f"TP (True Positive): {tp}")
    print(f"FP (False Positive): {fp}")
    print(f"FN (False Negative): {fn}")
    print(f"Recall: {recall:.2f}\n")


evaluate_threshold(p, y_true, 0.5)
evaluate_threshold(p, y_true, 0.3)

# 4.  Bank qaysi chegarani tanlashi kerak? Ikkala xatoning narxini solishtirib asoslang.  [2 ball]

# 5.  Nega bu masalada chiziqli regressiya emas, logistik regressiya ishlatiladi?  [1 ball]

Mijoz A uchun z = 0.40
Mijoz B uchun z = -4.000000
Mijoz A ning kreditni qaytarmaslik ehtimoli -> 0.60
Mijoz B ning kreditni qaytarmaslik ehtimoli -> 0.02
Mijoz A kreditni qaytarmaydi yani 1
Mijoz B kreditni qaytaradi yani 0
--- Chegara: 0.5 ---
TP (True Positive): 2
FP (False Positive): 0
FN (False Negative): 1
Recall: 0.67

--- Chegara: 0.3 ---
TP (True Positive): 3
FP (False Positive): 1
FN (False Negative): 0
Recall: 1.00



In [48]:
# M3. Taksi ilovasi: tarif tavsiyasi    KNN  •  10 ball
# Ilova yangi yo'lovchiga tarif tavsiya qiladi: Ekonom (E) yoki Komfort (K). Belgilar: o'rtacha safar masofasi (km) va oyiga safarlar soni.
# O'quv to'plami: Y1 (4, 4) E · Y2 (6, 7) K · Y3 (7, 5) K · Y4 (2, 3) E · Y5 (3, 2) E · Y6 (9, 9) K.
# Yangi yo'lovchi: (5, 5). Evklid masofasidan foydalaning.


# 1.  Yangi yo'lovchidan har bir nuqtagacha masofani hisoblang (2 xona aniqlikda).  [3 ball]

i = 1

for y in [[4, 4], [6, 7], [7, 5], [2, 3], [3, 2], [9, 9]]:
    print(f"Y{i} -> {round(np.sqrt(y[0]**2 + y[1]**2), 2)}")
    i = i + 1

# 2.  k = 1, k = 3 va k = 5 bo'lganda bashorat qaysi tarif?  [3 ball]
from sklearn.neighbors import KNeighborsClassifier

X = [[4, 4], [6, 7], [7, 5], [2, 3], [3, 2], [9, 9]]
y = ["E", "K", "K", "E", "E", "K"]

yangi_yolovchi = [[5, 5]]

for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    print(f"k = {k}: {knn.predict(yangi_yolovchi)[0]}")


# 3.  "Safarlar soni" o'rniga "oylik xarajat (so'm)" belgisi olinsa (qiymatlari 100 000 atrofida), KNN'ga nima bo'ladi va qanday tuzatiladi?  [2 ball]
#
# 4.  k ni to'g'ri tanlash uchun Pipeline va GridSearchCV bilan kod yozing (3–4 qator).  [2 ball]
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())])
grid = GridSearchCV(pipe, {"knn__n_neighbors": [1, 3, 5]}, cv=2)
grid.fit(X, y)
print(grid.best_params_)

Y1 -> 5.66
Y2 -> 9.22
Y3 -> 8.6
Y4 -> 3.61
Y5 -> 3.61
Y6 -> 12.73
k = 1: E
k = 3: K
k = 5: E
{'knn__n_neighbors': 1}


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 537, in __call__
    return estimator.score(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/pipeline.py", line 1202, in score
    return self.steps[-1][1].score(Xt, y, **score_params)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.fr

In [49]:
# M4. Poliklinika: diabet skriningi    Decision Tree  •  10 ball
# 20 ta bemor: 8 tasida diabet bor (1), 12 tasi sog'lom (0). Daraxt ikki bo'linishdan birini tanlashi kerak:
# A bo'linish: Glukoza ≤ 125 → chap [11 sog'lom, 2 diabet], o'ng [1 sog'lom, 6 diabet].
# B bo'linish: Yosh ≤ 45 → chap [7 sog'lom, 3 diabet], o'ng [5 sog'lom, 5 diabet].


# 1.  Ildiz tugunning gini qiymatini toping.  [1 ball]
gini_root = 1 - (12 / 20) ** 2 - (8 / 20) ** 2

print(gini_root)

# 2.  A bo'linish uchun har bir bolaning gini'sini, og'irlikli gini'ni va gain'ni toping.  [3 ball]
gini_left = 1 - (11 / 13) ** 2 - (2 / 13) ** 2
gini_right = 1 - (1 / 7) ** 2 - (6 / 7) ** 2

weighted_gini_A = (13 / 20) * gini_left + (7 / 20) * gini_right
gain_A = 0.48 - weighted_gini_A

print("Left Gini:", gini_left)
print("Right Gini:", gini_right)
print("Weighted Gini:", weighted_gini_A)
print("Gini Gain:", gain_A)

# 3.  B bo'linish uchun og'irlikli gini va gain'ni toping.  [2 ball]
gini_left = 1 - (7 / 10) ** 2 - (3 / 10) ** 2
gini_right = 1 - (5 / 10) ** 2 - (5 / 10) ** 2

weighted_gini_B = (10 / 20) * gini_left + (10 / 20) * gini_right
gain_B = 0.48 - weighted_gini_B

print("Left Gini:", gini_left)
print("Right Gini:", gini_right)
print("Weighted Gini:", weighted_gini_B)
print("Gini Gain:", gain_B)

# 4.  Daraxt qaysi bo'linishni tanlaydi va nima uchun?  [1 ball]
# Daraxt qaysi bo‘linishni tanlaydi? (1 ball)
# - A bo‘linish: Gain = 0.225
# - B bo‘linish: Gain = 0.02
# Javob: A bo‘linish tanlanadi, chunki uning Gini Gain qiymati kattaroq. Demak, Glukoza belgisi ma'lumotlarni diabet holati bo‘yicha yaxshiroq ajratadi.

# 5.  max_depth = 12 bo'lgan daraxt train'da 1.00, test'da 0.68 oldi. Muammo nima? Uni kamaytiradigan ikkita parametrni ayting.  [2 ball]
# Overfittingni kamaytiradigan ikkita parametr:
# 1. max_depth — daraxtning maksimal chuqurligini kamaytiradi.
# 2.  — har bir bargda bo‘lishi kerak bo‘lgan minimal namunalar sonini oshiradi.

# 6.  A bo'linishning o'ng bargini shifokor tushunadigan oddiy tilda qoida qilib yozing.  [1 ball]

0.48
Left Gini: 0.2603550295857988
Right Gini: 0.24489795918367352
Weighted Gini: 0.254945054945055
Gini Gain: 0.225054945054945
Left Gini: 0.42000000000000004
Right Gini: 0.5
Weighted Gini: 0.46
Gini Gain: 0.019999999999999962


In [ ]:
# M5. Mobil operator: mijoz ketishi    Random Forest va baholash  •  10 ball

# Operatorning 2000 mijozidan 300 tasi ketgan (churn = 1). Uchta model natijasi:

# Decision Tree (cheklovsiz): train 1.00 · test 0.79 · CV 0.78
# Decision Tree (max_depth = 5): train 0.88 · test 0.86 · CV 0.86

# Random Forest (200 daraxt): train 1.00 · test 0.89 · CV 0.88
# Random Forest test'da (400 mijoz, 60 tasi ketgan): TP = 36, FN = 24, FP = 20, TN = 320.


# 1.  "Hech kim ketmaydi" deydigan baseline accuracy'ni hisoblang va har bir modelni u bilan solishtiring.  [2 ball]

# "Hech kim ketmaydi" deydigan baseline accuracy' ni hisoblasak jami 1700 ta odam ketmagan boladi va bu 1700/2000 * 100 = 85% aniqlik chiqadi-> random forest va decision tree bilan solishtirganimizda bu yaxshi modeldek tuyiladi lekin bu hato


# 2.  Qaysi model overfitting qilgan? Random Forest'ning train 1.00 natijasi ham overfitting'mi? Izohlang.  [2 ball]


# 3.  Random Forest uchun precision, recall va F1 ni toping.  [3 ball]
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = [1] * 36 + [1] * 24 + [0] * 20 + [0] * 320
y_pred = [1] * 36 + [0] * 24 + [1] * 20 + [0] * 320

print(f"Precision: {precision_score(y_true, y_pred)}")
print(f"Recall: {recall_score(y_true, y_pred)}")
print(f"F1-score: {f1_score(y_true, y_pred)}")


# 4.  Eng muhim belgilar: oylik_tolov 0.31, shartnoma_turi 0.24, supportga_qongiroqlar 0.18. Marketing bo'limi: "supportga qo'ng'iroq qilganlar ketadi, support raqamini yopamiz". Bu xulosa to'g'rimi?  [1 ball]


# 5.  Uchala modelning CV natijasini cross_val_score bilan hisoblab, jadval (DataFrame) qiladigan kodni yozing (3–4 qator).  [2 ball]
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Decision Tree (depth=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}
results = {
    name: cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    for name, model in models.items()
}
df_results = pd.DataFrame(
    {
        "Model": results.keys(),
        "CV Accuracy": [scores.mean() for scores in results.values()],
    }
)
print(df_results)

Precision: 0.6428571428571429
Recall: 0.6
F1-score: 0.6206896551724138


NameError: name 'X_train' is not defined